In [76]:
# Import relevant libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import zipfile
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from datetime import datetime

In [77]:
# Current working directory
current_dir = os.getcwd()
print(current_dir)

/workspaces/DSE3101-Project/backend


In [78]:
# Go up to repo root, then into data folder to retrive raw data
zip_path = os.path.join(current_dir, "..", "data", "raw", "ResaleFlatPrices.zip")
zip_path

'/workspaces/DSE3101-Project/backend/../data/raw/ResaleFlatPrices.zip'

In [79]:
all_dfs = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
    
    for csv_file in csv_files:
        with zip_ref.open(csv_file) as f:
            df = pd.read_csv(f)
            all_dfs.append(df)

# Combine into one DataFrame
combined_df = pd.concat(all_dfs, ignore_index=True)

In [80]:
print("=" * 50)
print("STEP 1: INITIAL DATA INSPECTION")
print("=" * 50)

print(f"\nDataset shape: {combined_df.shape}")
print(f"\nColumn names:\n{combined_df.columns.tolist()}")
print(f"\nData types:\n{combined_df.dtypes}")
print(f"\nFirst few rows:\n{combined_df.head()}")
print(f"\nBasic statistics:\n{combined_df.describe()}")
print(f"\nUnique values per column:")
for col in combined_df.columns:
    print(f"  {col}: {combined_df[col].nunique()}")

STEP 1: INITIAL DATA INSPECTION

Dataset shape: (970969, 11)

Column names:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']

Data types:
month                      str
town                       str
flat_type                  str
block                      str
street_name                str
storey_range               str
floor_area_sqm         float64
flat_model                 str
lease_commence_date      int64
resale_price           float64
remaining_lease         object
dtype: object

First few rows:
     month        town flat_type block       street_name storey_range  \
0  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
1  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06   
2  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
3  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09   
4  1990-01  A

In [81]:
# get current year 
current_year = datetime.now().year  

# Extract year from lease_commence_date and calculate
combined_df['lease_year'] = pd.to_datetime(combined_df['lease_commence_date']).dt.year
combined_df['remaining_lease'] = 99 - (current_year - combined_df['lease_year'])

# Clean up
combined_df = combined_df.drop('lease_year', axis=1)

print("✓ Added remaining_lease column")
print(f"Sample: {combined_df[['lease_commence_date', 'remaining_lease']].head()}")


✓ Added remaining_lease column
Sample:    lease_commence_date  remaining_lease
0                 1977               43
1                 1977               43
2                 1977               43
3                 1977               43
4                 1976               43


In [82]:
print("\n" + "=" * 80)
print("STEP 2: MISSING VALUE ANALYSIS")
print("=" * 80)

missing_summary = pd.DataFrame({
    'Missing_Count': combined_df.isnull().sum(),
    'Percentage': (combined_df.isnull().sum() / len(combined_df)) * 100,
    'Data_Type': combined_df.dtypes
})
print(missing_summary)

# Handle missing values
if combined_df.isnull().sum().sum() > 0:
    # For critical columns, drop rows
    critical_cols = ['resale_price', 'floor_area_sqm', 'lease_commence_date', 'town', 'flat_type']
    combined_df = combined_df.dropna(subset=critical_cols)
    print(f"\n✓ Dropped rows with missing critical values")
    print(f"New shape: {combined_df.shape}")


STEP 2: MISSING VALUE ANALYSIS
                     Missing_Count  Percentage Data_Type
month                            0         0.0       str
town                             0         0.0       str
flat_type                        0         0.0       str
block                            0         0.0       str
street_name                      0         0.0       str
storey_range                     0         0.0       str
floor_area_sqm                   0         0.0   float64
flat_model                       0         0.0       str
lease_commence_date              0         0.0     int64
resale_price                     0         0.0   float64
remaining_lease                  0         0.0     int32


In [86]:
print("\n" + "=" * 80)
print("STEP 3: DUPLICATE REMOVAL")
print("=" * 80)

initial_rows = len(combined_df)
duplicates = combined_df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

print(combined_df[combined_df.duplicated(keep=False)].head())

if duplicates > 0:
    combined_df = combined_df.drop_duplicates()
    print(f"✓ Removed {initial_rows - len(combined_df)} duplicate rows")
    print(f"New shape: {combined_df.shape}")


STEP 3: DUPLICATE REMOVAL
Duplicate rows found: 2006
       month         town flat_type block    street_name storey_range  \
672  1990-01      GEYLANG    3 ROOM    47     CIRCUIT RD     01 TO 03   
673  1990-01      GEYLANG    3 ROOM    47     CIRCUIT RD     01 TO 03   
725  1990-01      HOUGANG    3 ROOM   308  HOUGANG AVE 5     10 TO 12   
726  1990-01      HOUGANG    3 ROOM   308  HOUGANG AVE 5     10 TO 12   
842  1990-01  JURONG WEST    3 ROOM   145    HU CHING RD     04 TO 06   

     floor_area_sqm      flat_model  lease_commence_date  resale_price  \
672            56.0        STANDARD                 1969       18000.0   
673            56.0        STANDARD                 1969       18000.0   
725            67.0  NEW GENERATION                 1983       47000.0   
726            67.0  NEW GENERATION                 1983       47000.0   
842            64.0        IMPROVED                 1976       23400.0   

     remaining_lease  
672               43  
673             

In [91]:
# ============================================
# STEP 4: DATA TYPE CONVERSION & PARSING
# ============================================
print("\n" + "=" * 80)
print("STEP 4: DATA TYPE CONVERSION & PARSING")
print("=" * 80)

# Convert month to datetime
combined_df['month'] = pd.to_datetime(combined_df['month'], format='%Y-%m')
print("✓ Converted 'month' to datetime")

# Find middle of storey_range (e.g., Split "01 TO 05" → get both numbers → average → single storey_mid column)
split_storey = combined_df['storey_range'].str.split(' TO ', expand=True).astype(int)
combined_df['storey_mid'] = split_storey[[0,1]].mean(axis=1)

# Ensure numeric columns are numeric
numeric_cols = ['block', 'floor_area', 'lease_commence', 'resale_price']
for col in numeric_cols:
    if col in combined_df.columns:
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

print("✓ Ensured numeric columns have correct data types")


STEP 4: DATA TYPE CONVERSION & PARSING
✓ Converted 'month' to datetime
✓ Ensured numeric columns have correct data types


In [ ]:
combined_df['storey_mid']